In [ ]:
import json
import os
from dotenv import load_dotenv
import urllib
from urllib import request

In [24]:
from dataclasses import dataclass
@dataclass
class Response:
    """Structured response from LLM calls."""
    content: list[dict[str, str]]
    usage: dict | None = None

In [25]:
class AnthropicLLM:
    def __init__(
        self,
        model: str,
        base_url: str = "https://api.anthropic.com/v1/messages",
    ):
        """Initialize the LLM with the given model"""
        load_dotenv()
        self.model = model
        self.base_url = base_url
        self.api_key = os.getenv("ANTHROPIC_API_KEY")

    def generate(
        self, messages: list[dict]) -> Response:

        body = {
            "model": self.model,
            "max_tokens": 1024,
            "messages": messages
        }

        request = urllib.request.Request(
            self.base_url,
            data=json.dumps(body).encode(),
            headers={
                "content-type": "application/json",
                "x-api-key": self.api_key,
                "anthropic-version": "2023-06-01"
            },
        )
        with urllib.request.urlopen(request) as response:
            data = json.loads(response.read())
  
        message = data["content"][-1]["text"]
        usage = data["usage"]
            # Format as Response dataclass
        return Response(
            content  = message,
            usage = usage,
        )

In [26]:
class Memory:
    """Simple memory module to store conversation history."""

    def __init__(self):
        self.messages = []

    def add(self, role: str, content: str, **kwargs) -> None:
        """Add a message to memory."""
        self.messages.append({"role": role, "content": content})

    def get_messages(self) -> list[dict]:
        """Get all messages."""
        return self.messages

In [ ]:
class TinyAgent:
    """A minimal, modular, and educational agent framework."""

    def __init__(self, llm: AnthropicLLM, memory: Memory):
        self.llm = llm
        self.memory = memory  # Add Memory


    def run(self, task: str) -> str:
        """Run the agent on a task."""

        # 1. Add the user's message
        self.memory.add("user", task)

        # For now, 'run' = one step, then return
        response_str = self._step()

        return response_str


    def _step(self) -> str:
        """Perform a single step."""

        # 2. Get the full conversation, including latest
        all_messages = self.memory.get_messages()

        # 3. Ask the LLM to reply
        try:
            response = self.llm.generate(all_messages)
        finally:
        # 4. Add the reply to memory
            self.memory.add("assistant", response.content)

        # 5. Return the latest response for display
        return response.content

In [28]:
# @markdown - Simple chat interface (dark theme)
# @markdown - Don't need to read/understand the GUI code
import html
import traceback
import ipywidgets as widgets
from IPython.display import display, HTML

# Render the model's markdown as HTML (pip install markdown); falls back to plain text
try:
    import markdown as md
    def to_html(text):
        return md.markdown(text, extensions=["fenced_code", "tables"])
except ImportError:
    def to_html(text):
        return f'<div style="white-space: pre-wrap;">{html.escape(text)}</div>'

# Palette: soft charcoal background + off-white text (pure #000/#fff is harsher on the eyes)
BG, PANEL, BORDER = "#1e1f24", "#2a2c33", "#3a3d45"
TEXT, MUTED = "#e6e6e6", "#9aa0a6"
USER_ACCENT, ASSISTANT_ACCENT = "#7dd3fc", "#a3e635"

CHAT_CSS = f"""
<style>
.tiny-chat {{ background:{BG}; color:{TEXT}; padding:16px 20px; border-radius:10px;
  font-family:-apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; font-size:14px; line-height:1.6;
  max-height:600px; overflow-y:auto; }}
.tiny-chat .msg {{ margin:0 0 14px 0; padding:10px 14px; border-radius:8px; background:{PANEL}; }}
.tiny-chat .msg.user {{ border-left:3px solid {USER_ACCENT}; }}
.tiny-chat .msg.assistant {{ border-left:3px solid {ASSISTANT_ACCENT}; }}
.tiny-chat .msg.err {{ border-left:3px solid #f87171; }}
.tiny-chat .msg:last-child {{ margin-bottom:0; }}
.tiny-chat .role {{ font-size:11px; font-weight:600; letter-spacing:.06em; text-transform:uppercase; margin-bottom:4px; }}
.tiny-chat .user .role {{ color:{USER_ACCENT}; }}
.tiny-chat .assistant .role {{ color:{ASSISTANT_ACCENT}; }}
.tiny-chat .err .role {{ color:#f87171; }}
.tiny-chat p, .tiny-chat li, .tiny-chat h1, .tiny-chat h2, .tiny-chat h3, .tiny-chat h4 {{ color:{TEXT}; }}
.tiny-chat p {{ margin:0 0 8px 0; }}
.tiny-chat p:last-child {{ margin-bottom:0; }}
.tiny-chat code {{ background:#16171b; color:#f0c674; padding:1px 5px; border-radius:4px; }}
.tiny-chat pre {{ background:#16171b; padding:10px; border-radius:6px; overflow-x:auto; }}
.tiny-chat pre code {{ padding:0; color:{TEXT}; }}
.tiny-chat a {{ color:{USER_ACCENT}; }}
.tiny-chat table {{ border-collapse:collapse; }}
.tiny-chat th, .tiny-chat td {{ border:1px solid {BORDER}; padding:4px 8px; }}
.tiny-chat .muted {{ color:{MUTED}; font-style:italic; }}
.tiny-chat .error {{ color:#fca5a5; white-space:pre-wrap; font-family:monospace; font-size:12px; }}
/* VS Code paints widget outputs white; make the container behind this UI transparent */
.cell-output-ipywidget-background:has(.tiny-chat-ui) {{ background-color: transparent !important; }}
/* Dark container for the whole UI + recolor built-in widget parts (labels, progress track, borders) */
.tiny-chat-ui {{ background:{BG}; padding:12px; border-radius:12px;
  --jp-widgets-color:{TEXT}; --jp-widgets-label-color:{TEXT}; --jp-ui-font-color1:{TEXT};
  --jp-layout-color2:{PANEL}; --jp-border-color1:{BORDER}; }}
.tiny-chat-ui .widget-label {{ color:{TEXT} !important; }}
</style>
"""


class ChatUI:
    """A simple chat interface for the agent."""

    def __init__(self, agent):
        self.agent = agent

        # Output widget to display the conversation
        self.chat_output = widgets.Output()

        # Progress bar for context window
        self.context_progress = widgets.IntProgress(
            value=0,
            min=0,
            max=8192,  # OLLAMA_CONTEXT_LENGTH, set in Setup
            description='Context:',
            bar_style='success',
            orientation='horizontal',
            layout=widgets.Layout(width='30%')
        )

        # Clear button to reset the conversation
        self.clear_button = widgets.Button(
            description='Clear',
            tooltip='Clear conversation',
            layout=widgets.Layout(width='100px', margin='0 0 0 10px'),
            style={'button_color': '#7f1d1d', 'text_color': TEXT},
        )
        self.clear_button.on_click(self._on_clear_clicked)

        # Text input widget for the user's message
        self.text_input = widgets.Textarea(
            value='',
            placeholder='Type your message here...',
            layout=widgets.Layout(width='80%', height='80px'),
            style={'background': PANEL, 'text_color': TEXT, 'font_size': '14px'},
        )

        # Button to send the message
        self.send_button = widgets.Button(
            description='Send',
            tooltip='Send message',
            layout=widgets.Layout(width='100px', margin='0 0 0 10px'),
            style={'button_color': '#0369a1', 'text_color': TEXT},
        )
        self.send_button.on_click(self._on_send_clicked)

    def _format_chat_history(self, messages, extra_html=""):
        parts = []
        for msg in messages:
            role = 'user' if msg['role'] == 'user' else 'assistant'
            parts.append(
                f'<div class="msg {role}"><div class="role">{role}</div>{to_html(str(msg["content"]))}</div>'
            )
        if not parts and not extra_html:
            parts.append('<div class="muted">Start the conversation below.</div>')
        return CHAT_CSS + '<div class="tiny-chat">' + "".join(parts) + extra_html + '</div>'

    def _render(self, extra_html="", messages=None):
        messages = self.agent.memory.get_messages() if messages is None else messages
        with self.chat_output:
            self.chat_output.clear_output()
            display(HTML(self._format_chat_history(messages, extra_html)))

    def _on_clear_clicked(self, b):
        self.agent.memory.messages = []
        self.context_progress.value = 0
        self.context_progress.bar_style = 'success'
        self._render('<div class="muted">Conversation cleared.</div>')

    def _on_send_clicked(self, b):
        user_msg = self.text_input.value.strip()
        if not user_msg:
            return

        # Clear input
        self.text_input.value = ''

        # Show "Responding..." state immediately
        temp_messages = self.agent.memory.get_messages() + [{'role': 'user', 'content': user_msg}]
        thinking = '<div class="msg assistant"><div class="role">assistant</div><div class="muted">Responding... ⏳</div></div>'
        self._render(thinking, messages=temp_messages)

        # Run the agent; show errors in the chat instead of failing silently
        try:
            self.agent.run(user_msg)
        except Exception:
            # Drop the unanswered user turn so the next request doesn't send two user messages in a row
            if self.agent.memory.messages and self.agent.memory.messages[-1]['role'] == 'user':
                self.agent.memory.messages.pop()
            error = f'<div class="msg err"><div class="role">error</div><div class="error">{html.escape(traceback.format_exc())}</div></div>'
            self._render(error, messages=temp_messages)
            return

        # Estimate context usage (~4 chars per token)
        est_tokens = sum(len(str(m['content'])) // 4 for m in self.agent.memory.get_messages())
        self.context_progress.value = min(est_tokens, self.context_progress.max)

        if self.context_progress.value > self.context_progress.max * 0.9:
            self.context_progress.bar_style = 'danger'
        elif self.context_progress.value > self.context_progress.max * 0.75:
            self.context_progress.bar_style = 'warning'
        else:
            self.context_progress.bar_style = 'success'

        # Re-render the actual chat history with the response
        self._render()

    def show(self):
        """Display the UI."""
        self._render()
        ui = widgets.VBox([
            self.chat_output,
            widgets.HBox([self.text_input, self.send_button], layout=widgets.Layout(align_items='center', margin='10px 0 0 0')),
            widgets.HBox([self.context_progress, self.clear_button], layout=widgets.Layout(align_items='center', margin='10px 0 0 0'))
        ])
        ui.add_class('tiny-chat-ui')  # hook for the dark-container CSS above
        display(ui)


In [29]:
Agent = TinyAgent(llm=AnthropicLLM(model = "claude-opus-5-5"), memory = Memory())

In [30]:
chat_ui = ChatUI(Agent)

In [31]:
chat_ui.show()